# Q4 - Offline Evaluation Harness

Re-ranking evaluation (AUC, MRR, nDCG@5/@10) over each impression's own
`article_ids_inview`, plus beyond-accuracy metrics (intra-list diversity,
novelty, coverage), cold-start-vs-warm and head-vs-tail slicing, and
bootstrap 95% CIs -- for both BM25 (Q2) and embeddings (Q3), per SPEC.md's
Q4 section. This is a different framing from Q2/Q3's candidate-generation
recall@K: see SPEC.md Q4 #1 for why a `score_inview` re-scoring adapter is
needed instead of just reusing the persisted top-200 lists.

Run top-to-bottom (or via `python evaluation_harness.py`) to rebuild
`data/processed/{dataset}/eval_metrics.json`.

Loading/filtering uses `polars` throughout (same reasoning as
`src/build_pipeline.ipynb`/`src/bm25_retrieval.ipynb`), with the same
`BUILD_LARGE_ONLY` flag convention and progress appended to
`build_progress.log`. Requires Q2's `bm25_topk.parquet`/`bm25_metrics.json`
and Q3's `article_embeddings.parquet` to already exist for every dataset in
scope.

## Setup

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset, normalize_rows
from cs4406m26_assignment1c1.evaluation import (
    auc_impression,
    mrr,
    ndcg_at_k,
    bootstrap_ci,
    intra_list_diversity,
    novelty,
    coverage,
)


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"
CHECKPOINT_DIR = DATA_DIR / "_eval_checkpoints"
# Rows per checkpointed chunk in evaluate_ranking (see that cell). Kept small
# because the WinError 10055 kernel-death this notebook keeps hitting on
# this machine turned out not to be tied to a fixed row count or even a
# fixed elapsed time -- 3,000,000- and 1,000,000-row chunk sizes both
# crashed at almost exactly their own chunk boundary despite very different
# wall-clock durations to reach it, and a local benchmark ruled out
# DataFrame-construction/write time as the trigger (~6s for 1M rows). Given
# no reliable root cause after multiple mitigations (see SPEC.md Q4 #9),
# the practical fix is bounding how much work any single crash can cost and
# retrying automatically (`_run_nbconvert_with_retries.py`) rather than
# chasing the exact mechanism further.
CHUNK_SIZE = 200_000

# Same flag/convention as src/build_pipeline.ipynb and src/bm25_retrieval.ipynb.
BUILD_LARGE_ONLY = True
_DEFAULT_DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)
# EVAL_DATASETS env var (comma-separated) restricts this run to a subset --
# lets the notebook be invoked once per dataset (nbconvert launches a fresh
# kernel per invocation) instead of always loading both large datasets'
# feature stores/BM25 indexes/embedding matrices into one process at once.
# Found necessary via a live memory trace during repeated WinError 10055
# kernel-death investigations: holding both ebnerd_large and mind_large
# simultaneously pinned free RAM at ~0.3GB out of 15.7GB total on this
# machine, well before ebnerd_large's 12.5M-row test split even started
# scoring -- the crash is a symptom of genuine memory exhaustion, not random
# Windows socket flakiness. Unset -> unchanged default (both datasets).
_env_datasets = os.environ.get("EVAL_DATASETS")
DATASETS = _env_datasets.split(",") if _env_datasets else _DEFAULT_DATASETS
METHODS = ["bm25", "embedding"]
SPLITS = ["val", "test"]


def dataset_fully_cached(name: str) -> bool:
    # True once evaluate_ranking's checkpoint already exists for all 4
    # (split, method) combos -- at that point nothing ever calls
    # score_inview_adapters[name] again (evaluate_ranking returns straight
    # from the checkpoint file), so rebuilding the BM25 index, embedding
    # matrix, and adapters for that dataset is pure wasted memory. Found
    # necessary after Windows Event Viewer showed dwm.exe/Explorer.exe
    # themselves crashing with STATUS_FATAL_MEMORY_EXHAUSTION (0xC00001AD)
    # at the same moment this notebook died -- confirming genuine
    # system-wide memory exhaustion, not a Jupyter-specific quirk, so every
    # avoidable allocation on a resume matters.
    return all((CHECKPOINT_DIR / name / f"{split}_{method}.parquet").exists() for split in SPLITS for method in METHODS)


RECENT_N_CLICKS = 20
NDCG_K_VALUES = [5, 10]
TOP_LIST_K = 10  # matches nDCG@10's window, per SPEC.md Q4 #3
HEAD_FRACTION = 0.2
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 0


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


log_progress(f"evaluation_harness started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY}, datasets={DATASETS})")

fully_cached = {name: dataset_fully_cached(name) for name in DATASETS}
if any(fully_cached.values()):
    log_progress(f"  fully cached, skipping BM25/embedding setup for: {[n for n in DATASETS if fully_cached[n]]}")

feature_store = {}
for name in DATASETS:
    feature_store[name] = {
        "articles": pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract", "category"]),
        "behaviors": pl.read_parquet(DATA_DIR / name / "behaviors.parquet", columns=["user_id", "article_ids_inview", "article_ids_clicked", "split"]),
        "history": pl.read_parquet(DATA_DIR / name / "history.parquet", columns=["user_id", "article_id_sequence"]),
    }
    log_progress(
        f"  {name}: loaded articles/behaviors/history "
        f"({feature_store[name]['articles'].height}, {feature_store[name]['behaviors'].height}, "
        f"{feature_store[name]['history'].height} rows)"
    )

_datasets_needing_scoring = [name for name in DATASETS if not fully_cached[name]]

embedding_paths = {name: DATA_DIR / name / "article_embeddings.parquet" for name in _datasets_needing_scoring}
missing = [str(p) for p in embedding_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"missing article_embeddings.parquet: {missing}. "
        "Run src/compute_embeddings_kaggle.ipynb on Kaggle first (see README.md)."
    )
embeddings_raw = {name: pl.read_parquet(embedding_paths[name]) for name in _datasets_needing_scoring}

# Rebuild BM25 indexes locally (cheap, seconds -- SPEC.md Q2 #1) rather than
# depending on bm25_retrieval.ipynb's in-memory state, which a separate
# notebook process can't see.
bm25_index = {}
title_by_id = {}
for name in DATASETS:
    if fully_cached[name]:
        bm25_index[name] = None
        title_by_id[name] = None
        continue
    articles = feature_store[name]["articles"]
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    doc_tokens = [tokenize(t) for t in texts]
    bm25_index[name] = build_index(articles["article_id"].to_list(), doc_tokens)
    title_by_id[name] = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))
    log_progress(f"  {name}: BM25 index rebuilt ({bm25_index[name].n_docs} docs)")

# Corpus embedding matrices, reindexed to articles.parquet order (same pattern as Q3).
corpus = {}
for name in DATASETS:
    if fully_cached[name]:
        corpus[name] = None
        continue
    articles = feature_store[name]["articles"]
    emb_lookup = dict(zip(
        embeddings_raw[name]["article_id"].to_list(),
        [np.asarray(v) for v in embeddings_raw[name]["embedding"].to_list()],
    ))
    doc_ids = articles["article_id"].to_numpy()
    missing_ids = set(doc_ids) - set(emb_lookup)
    if missing_ids:
        raise ValueError(f"{name}: {len(missing_ids)} articles have no embedding")
    matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
    corpus[name] = {"doc_ids": doc_ids, "matrix": matrix, "embedding_lookup": emb_lookup}
    log_progress(f"  {name}: embedding matrix built {matrix.shape}")

del embeddings_raw  # superseded by corpus[name]["matrix"]/["embedding_lookup"] above -- keeping both around duplicates every embedding's storage for no reason

{
    name: (
        "fully cached, setup skipped" if fully_cached[name]
        else {"bm25_docs": bm25_index[name].n_docs, "embedding_docs": len(corpus[name]["doc_ids"])}
    )
    for name in DATASETS
}

In [ ]:
def test_setup_aligned():
    for name in DATASETS:
        if fully_cached[name]:
            continue
        articles = feature_store[name]["articles"]
        assert bm25_index[name].n_docs == len(articles)
        assert corpus[name]["matrix"].shape[0] == len(articles)
        assert (corpus[name]["doc_ids"] == articles["article_id"].to_numpy()).all()
        assert not np.isnan(corpus[name]["matrix"]).any()


test_setup_aligned()
print("ok: BM25 indexes and embedding matrices rebuilt/loaded and aligned with articles.parquet (fully-cached datasets skipped -- nothing to align)")

## Metric-formula smoke tests

Hand-computed toy rankings with independently-derived expected values (SPEC.md
Q4 #2), before these functions are trusted against the real feature store.

In [3]:
def test_metric_formulas():
    # AUC: perfect ranking -> 1.0, inverted -> 0.0, all-tied -> 0.5 (this is
    # exactly what a cold-start user's all-zero BM25 score vector produces --
    # see the adapters below, so this case matters, not just a formula edge case)
    assert auc_impression([0.9, 0.1], [1, 0]) == 1.0
    assert auc_impression([0.1, 0.9], [1, 0]) == 0.0
    assert auc_impression([0.5, 0.5, 0.5], [1, 0, 1]) == 0.5

    # MRR: reciprocal rank of the best-scored clicked item
    assert mrr([0.9, 0.1, 0.05], [0, 1, 0]) == 0.5
    assert mrr([0.9, 0.1], [1, 0]) == 1.0

    # nDCG@5: hand-computed DCG/IDCG for a known 3-item ranking with 2 positives
    scores, labels = [0.9, 0.5, 0.1], [0, 1, 1]
    dcg = 0 / np.log2(2) + 1 / np.log2(3) + 1 / np.log2(4)
    idcg = 1 / np.log2(2) + 1 / np.log2(3)
    assert abs(ndcg_at_k(scores, labels, 5) - dcg / idcg) < 1e-9
    assert ndcg_at_k([0.9, 0.1], [1, 0], 5) == 1.0

    # bootstrap CI: point estimate is the plain mean; CI collapses to the point
    # for a constant array; CI always brackets the point for non-constant input
    point, lo, hi = bootstrap_ci([1.0, 1.0, 1.0], seed=0)
    assert point == lo == hi == 1.0
    point, lo, hi = bootstrap_ci([0.0, 1.0], n_iterations=2000, seed=0)
    assert lo <= point <= hi

    # intra-list diversity: 0 for <2 items or an all-same-category list, 1 for all-different
    cat = {"a": "x", "b": "x", "c": "y"}
    assert intra_list_diversity(["a"], cat) == 0.0
    assert intra_list_diversity(["a", "b"], cat) == 0.0
    assert intra_list_diversity(["a", "c"], cat) == 1.0

    # novelty: mean of precomputed per-article scores
    assert novelty(["a", "b"], {"a": 2.0, "b": 4.0}) == 3.0
    assert novelty([], {"a": 2.0}) == 0.0

    # coverage: union of retrieved sets over the catalog size
    assert coverage([["a", "b"], ["b", "c"]], 4) == 0.75


test_metric_formulas()
print("ok: AUC/MRR/nDCG/bootstrap_ci/intra_list_diversity/novelty/coverage match hand-computed values")

ok: AUC/MRR/nDCG/bootstrap_ci/intra_list_diversity/novelty/coverage match hand-computed values


## Cold-start user set (cross-checked against Q2)

Same definition Q2/Q3 use (empty `article_id_sequence`), recomputed here
(separate notebook process) but cross-checked against Q2's already-persisted
`bm25_metrics.json` cold-start impression counts, which must match exactly.
Unlike Q2/Q3's recall@K, cold-start users are **not excluded** from ranking
metrics below -- see the adapters section for why.

In [4]:
def build_coldstart_users(dataset: str) -> set:
    history = feature_store[dataset]["history"]
    behaviors = feature_store[dataset]["behaviors"]
    eval_user_ids = set(behaviors.filter(pl.col("split").is_in(SPLITS))["user_id"].to_list())
    history_eval = history.filter(pl.col("user_id").is_in(list(eval_user_ids)))
    return set(history_eval.filter(pl.col("article_id_sequence").list.len() == 0)["user_id"].to_list())


coldstart_users = {name: build_coldstart_users(name) for name in DATASETS}
{name: len(coldstart_users[name]) for name in DATASETS}

{'mind_large': 10423}

In [5]:
def test_coldstart_users():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
        if not bm25_metrics_path.exists():
            continue
        bm25_metrics = json.loads(bm25_metrics_path.read_text())
        for split in SPLITS:
            expected = bm25_metrics["n_impressions"][split]["excluded_coldstart"]
            actual = behaviors.filter(
                (pl.col("split") == split) & (pl.col("user_id").is_in(list(coldstart_users[name])))
            ).height
            assert actual == expected, f"{name}/{split}: cold-start impression count diverged from Q2's bm25_metrics.json"


test_coldstart_users()
print("ok: cold-start user set matches Q2's already-persisted counts exactly")

ok: cold-start user set matches Q2's already-persisted counts exactly


## Train-split popularity (novelty lookup + head/tail articles)

Shared basis for both #3's novelty metric and #4's optional head-vs-tail
slice (SPEC.md Q4 #3/#4): `pop(item) = clicks_train(item) / total_train_clicks`,
add-one-smoothed for never-clicked items so `novelty = -log2(pop)` never hits
`log2(0)`. Head = top 20% of *ever-clicked* articles by train-click-count;
tail = everything else, including never-clicked articles.

In [6]:
def compute_train_click_counts(dataset: str) -> Counter:
    behaviors = feature_store[dataset]["behaviors"]
    train_clicked = behaviors.filter(pl.col("split") == "train")["article_ids_clicked"].to_list()
    counts = Counter()
    for clicked in train_clicked:
        counts.update(clicked)
    return counts


train_click_counts = {name: compute_train_click_counts(name) for name in DATASETS}
train_total_clicks = {name: sum(train_click_counts[name].values()) for name in DATASETS}

novelty_lookup = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    counts = train_click_counts[name]
    total = train_total_clicks[name]
    n_articles = len(articles)
    smoothed_zero_pop = 1.0 / (total + n_articles)
    novelty_lookup[name] = {}
    for aid in articles["article_id"].to_list():
        c = counts.get(aid, 0)
        pop = (c / total) if c > 0 else smoothed_zero_pop
        novelty_lookup[name][aid] = -np.log2(pop)

head_articles = {}
for name in DATASETS:
    ever_clicked = sorted(train_click_counts[name].items(), key=lambda kv: -kv[1])
    n_head = max(1, int(len(ever_clicked) * HEAD_FRACTION))
    head_articles[name] = {aid for aid, _ in ever_clicked[:n_head]}

log_progress(f"train-split popularity/novelty computed for {DATASETS}")

{
    name: {
        "never_clicked_pct": 100 * (1 - len(train_click_counts[name]) / len(feature_store[name]["articles"])),
        "head_articles_share_of_train_clicks_pct": 100
        * sum(train_click_counts[name][aid] for aid in head_articles[name])
        / train_total_clicks[name],
    }
    for name in DATASETS
}

{'mind_large': {'never_clicked_pct': 86.1191923265259,
  'head_articles_share_of_train_clicks_pct': 97.1688910909485}}

In [7]:
def test_train_popularity():
    for name in DATASETS:
        assert all(v >= 0 for v in novelty_lookup[name].values())
        assert set(novelty_lookup[name]) == set(feature_store[name]["articles"]["article_id"].to_list())
        assert head_articles[name].issubset(set(train_click_counts[name]))

    # sanity-check against SPEC.md Q4 #3/#4's already-verified real-data figures
    # (small tolerance -- exact figures depend on the pipeline's random-free but
    # not bit-identical aggregation order) -- only meaningful when these two
    # specific datasets are actually in scope this run (not the case under
    # BUILD_LARGE_ONLY, where DATASETS is ["ebnerd_large", "mind_large"]).
    if "mind" in DATASETS:
        mind_never_clicked_pct = 100 * (1 - len(train_click_counts["mind"]) / len(feature_store["mind"]["articles"]))
        assert abs(mind_never_clicked_pct - 90.2) < 1.0
        mind_head_share = 100 * sum(train_click_counts["mind"][a] for a in head_articles["mind"]) / train_total_clicks["mind"]
        assert abs(mind_head_share - 90.2) < 1.0
    if "ebnerd" in DATASETS:
        ebnerd_never_clicked_pct = 100 * (1 - len(train_click_counts["ebnerd"]) / len(feature_store["ebnerd"]["articles"]))
        assert abs(ebnerd_never_clicked_pct - 91.8) < 1.0


test_train_popularity()
print("ok: novelty lookup covers every article and is non-negative; head/tail figures match SPEC.md's verified numbers when in scope")

ok: novelty lookup covers every article and is non-negative; head/tail figures match SPEC.md's verified numbers when in scope


In [8]:
# Drop train-split rows from behaviors now that click-count stats are
# computed -- nothing downstream ever needs them again (evaluate_ranking
# only ever filters to SPLITS=["val","test"]), but at ebnerd_large's scale
# the train split alone is 10.38M of 24.63M total behavior rows, and was
# sitting resident in memory for the rest of the run for no reason. Found
# via a live memory trace during a WinError 10055 crash investigation: free
# RAM was pinned at ~0.3GB out of 15.7GB total for the entire scoring phase
# on repeated runs -- the crash is a symptom of genuine memory exhaustion
# under BUILD_LARGE_ONLY's two-large-dataset-simultaneously footprint, not
# random Windows socket flakiness.
for name in DATASETS:
    before = feature_store[name]["behaviors"].height
    feature_store[name]["behaviors"] = feature_store[name]["behaviors"].filter(pl.col("split").is_in(SPLITS))
    after = feature_store[name]["behaviors"].height
    log_progress(f"  {name}: dropped train-split behavior rows to reduce peak memory ({before} -> {after} rows)")

In [9]:
def test_train_rows_dropped():
    for name in DATASETS:
        assert (feature_store[name]["behaviors"]["split"] == "train").sum() == 0
        assert set(feature_store[name]["behaviors"]["split"].unique().to_list()) <= set(SPLITS)


test_train_rows_dropped()
print("ok: train-split behavior rows dropped from memory after computing click-count stats")

ok: train-split behavior rows dropped from memory after computing click-count stats


## score_inview adapters

Uniform signature (SPEC.md Q4 #1): `score_inview(user_id, article_ids_inview)
-> dict[article_id, float]`, one instance per (dataset, method). BM25
memoizes the *last* user's full-corpus score vector -- the ranking-metrics
loop below processes each split sorted by `user_id`, so consecutive calls for
the same user hit the cache and `get_scores` runs once per user (~5ms, per
Q2's benchmark), not once per impression.

Cold-start users (empty history) get an all-zero score vector from
`get_scores`/an empty dict from `cosine_similarity_subset` -- both produce a
well-defined (if uninformative, all-tied -> AUC 0.5) ranking rather than an
exclusion. That is what makes cold-start-vs-warm a genuine *slice* for Q4,
unlike Q2/Q3's recall@K, where a cold-start user has no query at all and must
be excluded outright.

In [ ]:
def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)


def make_score_inview_adapters(dataset: str) -> dict:
    id_to_idx = {aid: i for i, aid in enumerate(bm25_index[dataset].doc_ids)}
    history = feature_store[dataset]["history"]
    history_lookup = dict(zip(history["user_id"].to_list(), history["article_id_sequence"].to_list()))
    title_lookup = title_by_id[dataset]
    embedding_lookup = corpus[dataset]["embedding_lookup"]
    emb_matrix = corpus[dataset]["matrix"]
    emb_doc_ids = corpus[dataset]["doc_ids"]
    # Precomputed once per dataset, not per impression: an earlier version of
    # cosine_similarity_subset rebuilt this id_to_idx dict (and re-normalized
    # the corpus subset) on every call -- at ebnerd_large scale (12.5M+
    # impressions x a 125,541-doc corpus), that rebuild was the dominant cost
    # of the whole embedding-scoring pass. See embeddings.py's
    # cosine_similarity_subset docstring and SPEC.md Q4 #9.
    emb_id_to_idx = {aid: i for i, aid in enumerate(emb_doc_ids)}
    # emb_matrix is already float32 (cast once when corpus[] was built above)
    # -- .astype(np.float32) unconditionally copies regardless of whether the
    # dtype already matches, so calling it here silently duplicated the full
    # (125,541, 768) matrix (~385MB) on every call for no reason. Found while
    # tracking down a MemoryError in this exact function at ebnerd_large
    # scale -- normalize_rows() still needs to produce its own output array,
    # but there's no reason to pay for an extra redundant copy on the input.
    emb_corpus_unit = normalize_rows(emb_matrix)

    bm25_cache = {"user_id": None, "scores": None}

    def bm25_fn(user_id, article_ids_inview):
        if bm25_cache["user_id"] != user_id:
            seq = history_lookup.get(user_id, [])
            query_tokens = build_user_query_tokens(seq, title_lookup)
            bm25_cache["user_id"] = user_id
            bm25_cache["scores"] = get_scores(bm25_index[dataset], query_tokens)
        scores = bm25_cache["scores"]
        return {aid: float(scores[id_to_idx[aid]]) for aid in article_ids_inview}

    def embedding_fn(user_id, article_ids_inview):
        seq = history_lookup.get(user_id, [])
        query_vector = build_user_query_vector(seq, embedding_lookup)
        scored = cosine_similarity_subset(query_vector, emb_corpus_unit, emb_doc_ids, emb_id_to_idx, article_ids_inview)
        return {aid: scored.get(aid, 0.0) for aid in article_ids_inview}

    return {"bm25": bm25_fn, "embedding": embedding_fn}


# None for a fully-cached dataset: evaluate_ranking returns straight from its
# checkpoint for every (split, method) combo in that case and never calls
# score_fn at all, so building the adapters (a per-dataset history_lookup
# dict + a duplicate normalized embedding matrix) would be pure wasted
# memory -- see dataset_fully_cached's docstring above for why that matters
# on this machine.
score_inview_adapters = {name: (None if fully_cached[name] else make_score_inview_adapters(name)) for name in DATASETS}

In [ ]:
def test_score_inview_adapters():
    for name in DATASETS:
        if fully_cached[name]:
            continue
        behaviors = feature_store[name]["behaviors"]
        sample = behaviors.row(0, named=True)
        inview = list(sample["article_ids_inview"])

        for method in METHODS:
            fn = score_inview_adapters[name][method]
            scored = fn(sample["user_id"], inview)
            assert set(scored) == set(inview)
            assert all(np.isfinite(v) for v in scored.values())

        # cold-start user (empty history) -> all-zero (tied) scores, not a crash
        history = feature_store[name]["history"]
        coldstart_rows = history.filter(pl.col("article_id_sequence").list.len() == 0)
        if coldstart_rows.height > 0:
            coldstart_user = coldstart_rows.row(0, named=True)["user_id"]
            bm25_scored = score_inview_adapters[name]["bm25"](coldstart_user, inview)
            assert set(bm25_scored.values()) == {0.0}
            emb_scored = score_inview_adapters[name]["embedding"](coldstart_user, inview)
            assert set(emb_scored.values()) == {0.0}

        # BM25 cache correctness: repeated calls for the same user return identical scores
        fn = score_inview_adapters[name]["bm25"]
        assert fn(sample["user_id"], inview) == fn(sample["user_id"], inview)


test_score_inview_adapters()
print("ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie (fully-cached datasets skipped -- adapters not built)")

## Per-impression ranking metrics

AUC/MRR/nDCG@5/@10 over each impression's own `article_ids_inview`, scored
by `score_inview` -- the re-ranking framing (SPEC.md Q4 #1), not Q2/Q3's
full-catalog candidate generation. `article_ids_inview` never has a
zero-click or single-candidate impression (verified: min inview size is 5
for EB-NeRD / 2 for MIND, and every impression has >=1 click), so these
metrics are always defined -- no impressions are excluded here, unlike
Q2/Q3's recall@K. Each split is processed sorted by `user_id` so the BM25
adapter's cache is effective. Also records each impression's top-10
re-ranked list (`top10_ids`), reused by the beyond-accuracy metrics next.

In [ ]:
def evaluate_ranking(dataset: str, split: str, method: str) -> pl.DataFrame:
    # Checkpointed, chunked: three separate full-restart crashes landed at
    # the exact same row (12,400,000/12,566,385, ebnerd_large's test/bm25
    # pass) despite a real memory fix in between that should have moved the
    # crash point if memory pressure were the sole cause -- consistent with
    # a Windows/ipykernel long-running-kernel resource issue tied to elapsed
    # wall-clock time rather than data volume (see SPEC.md Q4 #9 for the
    # fuller writeup). Splitting each pass into CHUNK_SIZE-row chunks, each
    # persisted to its own parquet immediately, means a crash anywhere loses
    # at most one partial chunk's work instead of the whole pass -- a retry
    # (fresh kernel, same on-disk checkpoints) skips every already-done
    # chunk and only redoes what's missing.
    final_ckpt = CHECKPOINT_DIR / dataset / f"{split}_{method}.parquet"
    if final_ckpt.exists():
        log_progress(f"  {dataset}/{split}/{method}: loaded from checkpoint")
        return pl.read_parquet(final_ckpt)

    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors.filter(pl.col("split") == split).sort("user_id")
    score_fn = score_inview_adapters[dataset][method]
    coldstart = coldstart_users[dataset]
    head = head_articles[dataset]

    user_ids_col = split_behaviors["user_id"].to_list()
    inview_col = split_behaviors["article_ids_inview"].to_list()
    clicked_col = split_behaviors["article_ids_clicked"].to_list()
    n_rows = len(user_ids_col)
    log_progress(f"  {dataset}/{split}/{method}: scoring {n_rows} impressions")

    # Canonical string pool for top10_ids -- without this, each row's top-10
    # list holds freshly-materialized Python string objects from the arrow-
    # to-python conversion above (no automatic interning across rows even
    # for the same recurring article_id), so a chunk's real memory footprint
    # grows with how far into it we've gotten, unlike the numpy arrays below
    # (paid once, upfront, at chunk size).
    canonical_id = {aid: aid for aid in feature_store[dataset]["articles"]["article_id"].to_list()}

    chunk_dir = CHECKPOINT_DIR / dataset / f"{split}_{method}_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    chunk_bounds = list(range(0, n_rows, CHUNK_SIZE)) + [n_rows]
    n_chunks = len(chunk_bounds) - 1

    for c in range(n_chunks):
        start, end = chunk_bounds[c], chunk_bounds[c + 1]
        chunk_path = chunk_dir / f"chunk_{c:03d}.parquet"
        if chunk_path.exists():
            continue  # already computed in a prior (crashed) invocation of this same run

        size = end - start
        # Preallocated numpy arrays for the scalar metric columns instead of
        # a list of per-impression dicts -- at ebnerd_large's test-split
        # scale (12.5M impressions), a dict per row would cost several GB of
        # pure Python dict/object overhead beyond the actual values (the
        # same class of problem Q2/Q3's chunked-write fixes addressed at the
        # write step, here addressed at construction time instead). numpy
        # arrays store the same data with no per-element object overhead.
        out_auc = np.empty(size, dtype=np.float64)
        out_mrr = np.empty(size, dtype=np.float64)
        out_is_coldstart = np.empty(size, dtype=bool)
        out_is_head = np.empty(size, dtype=bool)
        out_ndcg = {k: np.empty(size, dtype=np.float64) for k in NDCG_K_VALUES}
        out_top10 = [None] * size

        for local_i, row_i in enumerate(range(start, end)):
            user_id, inview, clicked = user_ids_col[row_i], inview_col[row_i], clicked_col[row_i]
            inview_ids = list(inview)
            clicked_set = set(clicked)
            scored = score_fn(user_id, inview_ids)
            scores = np.array([scored[aid] for aid in inview_ids])
            labels = np.array([aid in clicked_set for aid in inview_ids])
            top_order = np.argsort(-scores, kind="stable")[:TOP_LIST_K]

            out_auc[local_i] = auc_impression(scores, labels)
            out_mrr[local_i] = mrr(scores, labels)
            out_is_coldstart[local_i] = user_id in coldstart
            out_is_head[local_i] = any(aid in head for aid in clicked_set)
            out_top10[local_i] = [canonical_id[inview_ids[idx]] for idx in top_order]
            for k in NDCG_K_VALUES:
                out_ndcg[k][local_i] = ndcg_at_k(scores, labels, k)

            if (row_i + 1) % 200_000 == 0:
                log_progress(f"    {dataset}/{split}/{method}: {row_i + 1}/{n_rows} impressions scored")

        chunk_columns = {
            "user_id": user_ids_col[start:end],
            "auc": out_auc,
            "mrr": out_mrr,
            "is_coldstart": out_is_coldstart,
            "is_head": out_is_head,
            "top10_ids": out_top10,
            "dataset": [dataset] * size,
            "split": [split] * size,
            "method": [method] * size,
        }
        for k in NDCG_K_VALUES:
            chunk_columns[f"ndcg{k}"] = out_ndcg[k]
        # Write-then-rename: a crash mid-write on the direct path can leave a
        # truncated parquet file that then reads back as "corrupt" on a later
        # resume (hit twice in practice on this exact final-merge step, see
        # below). os.replace is atomic on both Windows and POSIX, so the
        # visible chunk_path is always either absent or fully valid.
        tmp_path = chunk_path.with_suffix(".parquet.tmp")
        pl.DataFrame(chunk_columns).write_parquet(tmp_path)
        os.replace(tmp_path, chunk_path)
        log_progress(f"  {dataset}/{split}/{method}: chunk {c + 1}/{n_chunks} checkpointed ({start}-{end})")

    df = pl.concat([pl.read_parquet(chunk_dir / f"chunk_{c:03d}.parquet") for c in range(n_chunks)])
    final_ckpt.parent.mkdir(parents=True, exist_ok=True)
    tmp_final = final_ckpt.with_suffix(".parquet.tmp")
    df.write_parquet(tmp_final)
    os.replace(tmp_final, final_ckpt)
    shutil.rmtree(chunk_dir)
    log_progress(f"  {dataset}/{split}/{method}: done, checkpointed")
    return df


ranking_results = pl.concat(
    [evaluate_ranking(name, split, method) for name in DATASETS for split in SPLITS for method in METHODS]
)
ranking_results.shape

In [13]:
def test_ranking_metrics():
    expected_total = sum(
        feature_store[name]["behaviors"].filter(pl.col("split") == split).height
        for name in DATASETS for split in SPLITS
    ) * len(METHODS)
    assert len(ranking_results) == expected_total

    for col in ["auc", "mrr", "ndcg5", "ndcg10"]:
        assert ranking_results[col].is_between(0.0, 1.0).all()
        assert ranking_results[col].null_count() == 0

    for name in DATASETS:
        for split in SPLITS:
            n_total = feature_store[name]["behaviors"].filter(pl.col("split") == split).height
            for method in METHODS:
                subset = ranking_results.filter(
                    (pl.col("dataset") == name) & (pl.col("split") == split) & (pl.col("method") == method)
                )
                n_evaluated = subset.height
                n_excluded = 0  # ranking metrics are always defined -- see markdown above
                assert n_evaluated + n_excluded == n_total


test_ranking_metrics()
print("ok: ranking metrics computed for every impression (n_evaluated + n_excluded == n_total), all values in [0,1]")

ok: ranking metrics computed for every impression (n_evaluated + n_excluded == n_total), all values in [0,1]


## Beyond-accuracy metrics (diversity, novelty, coverage)

Intra-list diversity and novelty over each impression's top-10 re-ranked list
(SPEC.md Q4 #3). Coverage is read from Q2/Q3's persisted top-K artifacts, not
the reranked list, since it's a candidate-generation-framed statistic.

In [ ]:
category_lookup = {
    name: dict(zip(feature_store[name]["articles"]["article_id"].to_list(), feature_store[name]["articles"]["category"].to_list()))
    for name in DATASETS
}

top10_col = ranking_results["top10_ids"].to_list()
dataset_col = ranking_results["dataset"].to_list()
ild_values = [intra_list_diversity(top10, category_lookup[dataset]) for top10, dataset in zip(top10_col, dataset_col)]
novelty_values = [novelty(top10, novelty_lookup[dataset]) for top10, dataset in zip(top10_col, dataset_col)]

# top10_ids dropped once consumed: it's a List[str] (10 article IDs) column
# over every impression -- at ebnerd_large's ~28.5M-row combined val+test
# ranking_results, that's several GB nothing downstream reads again
# (compute_coverage reads from Q2/Q3's own persisted top-K parquet files,
# not from ranking_results). Found necessary after a Rust/polars allocator
# panic ("memory allocation of 80000000 bytes failed") during the bootstrap
# CI step right after this cell, on a machine already confirmed to be under
# genuine system-wide memory exhaustion (SPEC.md Q4 #9).
ranking_results = ranking_results.with_columns(
    pl.Series("ild", ild_values),
    pl.Series("novelty", novelty_values),
).drop("top10_ids")
del top10_col
log_progress("beyond-accuracy metrics (ild, novelty) computed")


def compute_coverage(dataset: str, method: str) -> float:
    topk_path = DATA_DIR / dataset / f"{method}_topk.parquet"
    topk_df = pl.read_parquet(topk_path)
    n_articles = len(feature_store[dataset]["articles"])
    return coverage(topk_df["retrieved_article_ids"].to_list(), n_articles)


coverage_metrics = {(name, method): compute_coverage(name, method) for name in DATASETS for method in METHODS}
coverage_metrics

In [15]:
def test_beyond_accuracy_metrics():
    assert ranking_results["ild"].is_between(0.0, 1.0).all()
    assert (ranking_results["novelty"] >= 0.0).all()
    for cov in coverage_metrics.values():
        assert 0.0 <= cov <= 1.0

    # identical-category list -> ILD == 0 (already unit-tested on intra_list_diversity
    # directly in the metric-formula smoke tests; confirms the column-building path agrees)
    assert intra_list_diversity(["a", "b"], {"a": "x", "b": "x"}) == 0.0


test_beyond_accuracy_metrics()
print("ok: intra-list diversity in [0,1], novelty >= 0, coverage in [0,1] for both methods")

ok: intra-list diversity in [0,1], novelty >= 0, coverage in [0,1] for both methods


## Slicing

Cold-start vs. warm (required) and head vs. tail (optional), per SPEC.md
Q4 #4. Head/tail is applied at the impression level: an impression counts as
`head` if at least one of its clicked articles is a head item (a single
popular click is enough to signal what the method needs to get right), else
`tail`.

In [16]:
SLICE_DEFINITIONS = {
    "overall": lambda df: pl.Series([True] * df.height),
    "cold_start": lambda df: df["is_coldstart"],
    "warm": lambda df: ~df["is_coldstart"],
    "head": lambda df: df["is_head"],
    "tail": lambda df: ~df["is_head"],
}
METRIC_COLUMNS = ["auc", "mrr", "ndcg5", "ndcg10", "ild", "novelty"]
list(SLICE_DEFINITIONS)

['overall', 'cold_start', 'warm', 'head', 'tail']

In [17]:
def test_slicing_partition():
    for _, group in ranking_results.group_by(["dataset", "split", "method"]):
        cold = SLICE_DEFINITIONS["cold_start"](group)
        warm = SLICE_DEFINITIONS["warm"](group)
        assert (cold | warm).all() and not (cold & warm).any()

        head = SLICE_DEFINITIONS["head"](group)
        tail = SLICE_DEFINITIONS["tail"](group)
        assert (head | tail).all() and not (head & tail).any()


test_slicing_partition()
print("ok: cold-start/warm and head/tail slices are complete, non-overlapping partitions")

ok: cold-start/warm and head/tail slices are complete, non-overlapping partitions


## Bootstrap 95% CI

Resamples impressions (SPEC.md Q4 #5), 1,000 iterations, per
`(dataset, method, split, slice, metric)`. EB-NeRD's 0-cold-start-user slice
is empty by construction (see #3) -- recorded as `None` rather than a
degenerate CI, stated plainly rather than hidden.

In [ ]:
def compute_bootstrap_metrics() -> dict:
    results = {}
    # Only the columns this function actually touches -- group_by would
    # otherwise carry every column of ranking_results (including user_id,
    # never read here) into each group for no reason.
    slim = ranking_results.select(["dataset", "split", "method", "is_coldstart", "is_head", *METRIC_COLUMNS])
    for (name, split, method), group in slim.group_by(["dataset", "split", "method"]):
        slices = {}
        for slice_name, mask_fn in SLICE_DEFINITIONS.items():
            # "overall"'s mask is `pl.Series([True] * df.height)` -- a full
            # duplicate of `group` for a no-op filter. Filtering per-metric-
            # column below (not the whole `group` DataFrame per slice, as
            # the previous version did) means every other slice only ever
            # materializes one column's worth of data too, not a full
            # multi-column copy. At ebnerd_large's test-split scale
            # (12.5M rows), `group` plus one "overall"-slice full copy were
            # two multi-hundred-MB-to-GB copies of nearly the same data
            # alive at once -- found while digging into a DeadKernelError
            # here on an already memory-strapped machine (SPEC.md Q4 #9).
            is_overall = slice_name == "overall"
            mask = None if is_overall else mask_fn(group)
            n_rows = group.height if is_overall else int(mask.sum())
            slice_metrics = {}
            for metric in METRIC_COLUMNS:
                if n_rows == 0:
                    slice_metrics[metric] = None
                    continue
                col_values = group[metric] if is_overall else group[metric].filter(mask)
                point, lo, hi = bootstrap_ci(
                    col_values.to_numpy(), n_iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED,
                    max_chunk_cells=20_000_000,
                )
                slice_metrics[metric] = {"point": point, "ci_lo": lo, "ci_hi": hi}
            slices[slice_name] = slice_metrics
        results.setdefault(name, {}).setdefault(method, {})[split] = slices
        log_progress(f"  bootstrap CIs computed for {name}/{method}/{split}")
    return results


bootstrap_metrics = compute_bootstrap_metrics()

In [19]:
def test_bootstrap_ci():
    n_checked = 0
    n_none = 0
    for methods in bootstrap_metrics.values():
        for splits in methods.values():
            for slices in splits.values():
                for metrics_dict in slices.values():
                    for stats in metrics_dict.values():
                        if stats is None:
                            n_none += 1
                            continue
                        assert stats["ci_lo"] <= stats["point"] <= stats["ci_hi"] + 1e-9
                        n_checked += 1
    assert n_checked > 0
    # expected empty slices: any dataset with zero cold-start users (currently
    # ebnerd and ebnerd_small, not mind) contributes one empty cold_start slice
    # per (metric, method, split) -- computed from coldstart_users, not hardcoded,
    # since which datasets have zero cold-start users is a data fact, not a constant
    n_zero_coldstart_datasets = sum(1 for name in DATASETS if len(coldstart_users[name]) == 0)
    expected_none = len(METRIC_COLUMNS) * len(METHODS) * len(SPLITS) * n_zero_coldstart_datasets
    assert n_none == expected_none, f"expected {expected_none} empty (zero-cold-start-dataset) slices, got {n_none}"
    return n_checked, n_none


n_checked, n_none = test_bootstrap_ci()
print(f"ok: bootstrap CIs bracket the point estimate for {n_checked} (metric, slice) combinations; {n_none} degenerate as expected")

ok: bootstrap CIs bracket the point estimate for 120 (metric, slice) combinations; 0 degenerate as expected


## Anti-gaming (Q9) confirmation

A schema-column assertion (SPEC.md Q4 #7), not new engineering: the unified
schema never carried EB-NeRD's look-ahead fields in the first place, so
there's nothing to toggle for a with/without-unavailable-features comparison.

In [20]:
def test_no_leakage_columns():
    expected_behaviors_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id", "split",
    }
    expected_history_cols = {
        "user_id", "dataset", "article_id_sequence", "timestamp_sequence",
        "read_time_sequence", "scroll_percentage_sequence",
    }
    forbidden = {"next_read_time", "next_scroll_percentage"}

    # Checked against the actual on-disk schema (not feature_store, which only
    # loads a column-projected subset above for memory reasons at ebnerd_large
    # scale) -- this is what actually matters for the anti-gaming claim.
    for name in DATASETS:
        behaviors_cols = set(pl.read_parquet_schema(DATA_DIR / name / "behaviors.parquet"))
        history_cols = set(pl.read_parquet_schema(DATA_DIR / name / "history.parquet"))
        assert behaviors_cols == expected_behaviors_cols
        assert history_cols == expected_history_cols
        assert forbidden.isdisjoint(behaviors_cols) and forbidden.isdisjoint(history_cols)


test_no_leakage_columns()
print(
    "ok: anti-gaming confirmed -- unified schema carries no look-ahead fields "
    "(next_read_time/next_scroll_percentage); nothing to toggle for a "
    "with/without-unavailable-features comparison"
)

ok: anti-gaming confirmed -- unified schema carries no look-ahead fields (next_read_time/next_scroll_percentage); nothing to toggle for a with/without-unavailable-features comparison


## Persist eval_metrics.json

`{method: {split: {slice: {metric: {point, ci_lo, ci_hi}}}}}` per dataset
(SPEC.md Q4 #8), plus coverage (candidate-generation-framed, not sliced) and
the anti-gaming confirmation.

In [ ]:
def write_eval_metrics(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    payload = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "recent_n_clicks": RECENT_N_CLICKS,
            "ndcg_k_values": NDCG_K_VALUES,
            "top_list_k": TOP_LIST_K,
            "head_fraction": HEAD_FRACTION,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        },
        "ranking_metrics": bootstrap_metrics[dataset],
        "coverage": {method: coverage_metrics[(dataset, method)] for method in METHODS},
        "anti_gaming_confirmed": True,
    }
    (out_dir / "eval_metrics.json").write_text(json.dumps(payload, indent=2))
    log_progress(f"  {dataset}: wrote eval_metrics.json")

    # Clean up this dataset's evaluate_ranking checkpoints now that the final
    # output is safely written -- they're a resume aid for an in-progress
    # run, not a deliverable, and leaving them around risks a future run
    # silently reusing stale per-chunk results after a code change.
    ckpt_dir = CHECKPOINT_DIR / dataset
    if ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)

    return out_dir


eval_out_dirs = {name: write_eval_metrics(name) for name in DATASETS}
log_progress("evaluation_harness: all outputs written")
eval_out_dirs

In [22]:
def test_eval_metrics_roundtrip():
    for name in DATASETS:
        path = eval_out_dirs[name] / "eval_metrics.json"
        assert path.exists()
        reloaded = json.loads(path.read_text())

        assert set(reloaded["ranking_metrics"]) == set(METHODS)
        for method in METHODS:
            assert set(reloaded["ranking_metrics"][method]) == set(SPLITS)
            for split in SPLITS:
                assert set(reloaded["ranking_metrics"][method][split]) == set(SLICE_DEFINITIONS)

        assert reloaded["coverage"]["bm25"] == coverage_metrics[(name, "bm25")]
        assert reloaded["coverage"]["embedding"] == coverage_metrics[(name, "embedding")]
        assert reloaded["anti_gaming_confirmed"] is True


test_eval_metrics_roundtrip()
log_progress("evaluation_harness completed successfully")
print("ok: eval_metrics.json persisted and round-trips for every dataset")

ok: eval_metrics.json persisted and round-trips for every dataset


# Manual Verification Complete